In [2]:
"""
Generate all four methodology figures in one run.

    python make_all_figures.py

Produces, in the current folder, PNG + PDF for each:
  1. fig_signal_transformation   (9.3, illustrative)  - preprocessing chain
  2. fig_psd_damage_shift        (9.3 / Results)      - undamaged vs damaged PSD
  3. fig_loss_comparison         (9.4 / Results)      - Frobenius vs sigma_min
  4. fig_singularity_vs_freq     (9.4, illustrative)  - singularity at resonance

Figures 1-3 use synthetic placeholder data so the script runs out of the box.
Search for the comments marked  >>> REPLACE  to swap in your real data.
Use the PDF outputs in LaTeX (vector, stays sharp at any size).

Requires: numpy, scipy, matplotlib.
"""
import numpy as np
from scipy.signal import welch
import matplotlib as mpl
import matplotlib.pyplot as plt


# ===========================================================================
# Shared style + palette
# ===========================================================================
COLORS = {
    "primary":   "#1f4e79",   # dark blue
    "secondary": "#c0392b",   # brick red
    "accent":    "#e08a1e",   # amber
    "muted":     "#7f7f7f",   # grey
    "light":     "#a9c4e0",   # light blue
    "dark":      "#222222",
}


def apply_style():
    mpl.rcParams.update({
        "font.family":       "serif",
        "font.serif":        ["DejaVu Serif", "Times New Roman", "Times"],
        "font.size":         11,
        "axes.titlesize":    12,
        "axes.labelsize":    11,
        "xtick.labelsize":   10,
        "ytick.labelsize":   10,
        "legend.fontsize":   10,
        "axes.linewidth":    0.9,
        "lines.linewidth":   1.4,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.6,
        "figure.dpi":        120,
        "savefig.dpi":       300,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.03,
    })


def save(fig, name):
    fig.savefig(name + ".png")
    fig.savefig(name + ".pdf")
    plt.close(fig)
    print(f"  saved {name}.{{png,pdf}}")


# ===========================================================================
# Small shear-building model (used by figures 3 and 4)
# ===========================================================================
def shear_matrices(n, k_storey=1.0e6, m_floor=1.0e3):
    """Tridiagonal shear-building stiffness K and lumped mass M for n storeys."""
    K = np.zeros((n, n))
    for i in range(n):
        K[i, i] += k_storey
        if i > 0:
            K[i, i] += k_storey
            K[i, i - 1] -= k_storey
            K[i - 1, i] -= k_storey
    M = np.eye(n) * m_floor
    return K, M


def storey_stiffness_components(n, k_storey=1.0e6):
    """Per-storey K_i so that K(alpha) = sum_i alpha_i K_i."""
    comps = []
    for s in range(n):
        Ki = np.zeros((n, n))
        i = s
        Ki[i, i] += k_storey
        if i > 0:
            Ki[i, i] += k_storey
            Ki[i, i - 1] -= k_storey
            Ki[i - 1, i] -= k_storey
        comps.append(Ki)
    return comps


def fundamental_w2(K, M):
    """Smallest generalised eigenvalue (omega^2) for SPD diagonal M."""
    Minv_sqrt = np.diag(1.0 / np.sqrt(np.diag(M)))
    A = Minv_sqrt @ K @ Minv_sqrt
    return np.linalg.eigvalsh(A).min()


# ===========================================================================
# Figure 1 — signal transformation through preprocessing (9.3, illustrative)
# ===========================================================================
def make_demo_signal(fs=1000.0, duration=40.0, seed=0):
    """>>> REPLACE with a loaded acceleration record for the real figure."""
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1.0 / fs)
    modes = [(2.3, 1.0), (6.1, 0.55), (9.8, 0.30), (13.2, 0.15)]
    x = np.zeros_like(t)
    for f0, amp in modes:
        phase = rng.uniform(0, 2 * np.pi)
        env = np.convolve(rng.standard_normal(len(t)), np.hanning(400),
                          mode="same")
        x += amp * env * np.sin(2 * np.pi * f0 * t + phase)
    x += 0.15 * rng.standard_normal(len(t))
    return t, x


def fig_signal_transformation():
    fs, nperseg, eps = 1000.0, 2048, 1e-10
    P_MIN, P_MAX = -10.00, -2.09          # >>> REPLACE with your training bounds

    t, accel = make_demo_signal(fs=fs)
    f, psd = welch(accel, fs=fs, nperseg=nperseg)
    psd_log = np.log10(psd + eps)
    psd_norm = np.clip((psd_log - P_MIN) / (P_MAX - P_MIN), 0, 1)

    fig, axes = plt.subplots(4, 1, figsize=(6.4, 8.2))

    ax = axes[0]
    win = slice(0, int(4 * fs))
    ax.plot(t[win], accel[win], color=COLORS["primary"], lw=0.7)
    ax.set_xlabel("Time [s]"); ax.set_ylabel(r"Accel. [m/s$^2$]")
    ax.set_title("(a) Raw acceleration record", loc="left")

    ax = axes[1]
    ax.plot(f, psd, color=COLORS["secondary"]); ax.set_xlim(0, 20)
    ax.set_xlabel("Frequency [Hz]"); ax.set_ylabel("PSD [linear]")
    ax.set_title("(b) Welch PSD (linear scale)", loc="left")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

    ax = axes[2]
    ax.plot(f, psd_log, color=COLORS["accent"]); ax.set_xlim(0, 20)
    ax.set_xlabel("Frequency [Hz]"); ax.set_ylabel(r"$\log_{10}$ PSD")
    ax.set_title(r"(c) After $\log_{10}$ compression", loc="left")

    ax = axes[3]
    ax.plot(f, psd_norm, color=COLORS["primary"])
    ax.set_xlim(0, 20); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Frequency [Hz]"); ax.set_ylabel("Normalised")
    ax.set_title("(d) After normalisation to [0, 1]", loc="left")
    ax.axhline(0, color=COLORS["muted"], lw=0.6, ls=":")
    ax.axhline(1, color=COLORS["muted"], lw=0.6, ls=":")

    fig.tight_layout(h_pad=1.4)
    save(fig, "fig_signal_transformation")


# ===========================================================================
# Figure 2 — undamaged vs damaged PSD overlay (9.3 / Results)
# ===========================================================================
def make_demo_psd(peak_freqs, fs=1000.0, nperseg=2048, seed=1):
    """>>> REPLACE with real welch() PSDs from undamaged/damaged records."""
    rng = np.random.default_rng(seed)
    f = np.linspace(0, fs / 2, nperseg // 2 + 1)
    psd = np.full_like(f, 1e-8)
    for f0, a in zip(peak_freqs, [1.0, 0.5, 0.28, 0.14]):
        psd += a * (0.05 ** 2) / ((f - f0) ** 2 + 0.05 ** 2)
    psd *= (1 + 0.05 * rng.standard_normal(len(f)))
    return f, np.abs(psd)


def fig_psd_damage_shift():
    undamaged_peaks = [2.30, 6.10, 9.80, 13.20]
    damaged_peaks = [2.05, 5.55, 9.10, 12.55]   # stiffness loss -> lower f

    f, psd_u = make_demo_psd(undamaged_peaks, seed=1)
    _, psd_d = make_demo_psd(damaged_peaks, seed=2)

    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.semilogy(f, psd_u, color=COLORS["primary"], label="Undamaged")
    ax.semilogy(f, psd_d, color=COLORS["secondary"], ls="--", label="Damaged")
    ax.set_xlim(0, 16)
    # Extra headroom above the tallest peak so the legend and annotation
    # do not collide with the data.
    ax.set_ylim(top=psd_u.max() * 60)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"Power spectral density [(m/s$^2$)$^2$/Hz], log scale")
    ax.legend(frameon=False, loc="upper right", ncol=2)

    # Peak-shift annotation on the fundamental mode, placed in clear space
    # above the peaks with the label offset to the left so it never overlaps.
    yb = psd_u.max() * 3
    ax.annotate("", xy=(damaged_peaks[0], yb), xytext=(undamaged_peaks[0], yb),
                arrowprops=dict(arrowstyle="<-", color=COLORS["dark"], lw=1.3))
    ax.annotate("peak shifts to\nlower frequency\nunder damage",
                xy=((undamaged_peaks[0] + damaged_peaks[0]) / 2, yb),
                xytext=(4.4, yb * 4),
                arrowprops=dict(arrowstyle="-", color=COLORS["muted"], lw=0.7),
                ha="left", va="center", fontsize=8.5, color=COLORS["dark"])
    for pf, c in [(undamaged_peaks[0], COLORS["primary"]),
                  (damaged_peaks[0], COLORS["secondary"])]:
        ax.axvline(pf, color=c, lw=0.6, ls=":", alpha=0.6)

    fig.tight_layout()
    save(fig, "fig_psd_damage_shift")


# ===========================================================================
# Figure 3 — Frobenius vs singular-value loss (9.4 / Results)
# ===========================================================================
def fig_loss_comparison():
    n = 4
    _, M = shear_matrices(n)
    comps = storey_stiffness_components(n)

    def K_of(alpha):
        return sum(a * Ki for a, Ki in zip(alpha, comps))

    # >>> REPLACE alpha_true / K_of / M with your real Johnson system if desired,
    # or load the Frobenius and sigma_min values your training code logs.
    alpha_true = np.array([0.6, 1.0, 1.0, 1.0])
    w2 = fundamental_w2(K_of(alpha_true), M)

    sweep = np.linspace(0.2, 1.0, 200)
    frob, ratio = [], []
    for a1 in sweep:
        R = K_of(np.array([a1, 1.0, 1.0, 1.0])) - w2 * M
        frob.append(np.linalg.norm(R, "fro"))
        sv = np.linalg.svd(R, compute_uv=False)
        ratio.append(sv.min() / sv.max())
    frob, ratio = np.array(frob), np.array(ratio)
    frob_n, ratio_n = frob / frob.max(), ratio / ratio.max()

    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.plot(sweep, frob_n, color=COLORS["muted"], ls="--",
            label=r"Frobenius residual $\|K(\hat{\alpha})-\omega^2 M\|_F$")
    ax.plot(sweep, ratio_n, color=COLORS["secondary"],
            label=r"Singular-value ratio $\sigma_{\min}/\sigma_{\max}$")
    ax.axvline(alpha_true[0], color=COLORS["primary"], lw=0.9, ls=":")
    # Label the true-alpha line directly rather than in the legend.
    ax.text(alpha_true[0] + 0.01, 0.5, r"true $\alpha_1$",
            color=COLORS["primary"], rotation=90, va="center", fontsize=9)
    ax.set_xlabel(r"Predicted storey-1 stiffness $\hat{\alpha}_1$")
    ax.set_ylabel("Physics-loss value (each normalised to its own maximum)")
    ax.set_ylim(-0.03, 1.20)
    ax.set_xlim(0.2, 1.0)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.02),
              fontsize=9.5)

    fig.tight_layout()
    save(fig, "fig_loss_comparison")
    print(f"    (Frobenius varies {(frob.max()-frob.min())/frob.max()*100:.1f}%,"
          f" ratio varies {(ratio.max()-ratio.min())/ratio.max()*100:.1f}%)")


# ===========================================================================
# Figure 4 — singularity vs frequency (9.4, illustrative)
# ===========================================================================
def fig_singularity_vs_freq():
    n = 4
    K, M = shear_matrices(n)
    Minv_sqrt = np.diag(1.0 / np.sqrt(np.diag(M)))
    fn = np.sqrt(np.linalg.eigvalsh(Minv_sqrt @ K @ Minv_sqrt)) / (2 * np.pi)

    f_sweep = np.linspace(0.1, fn.max() * 1.15, 2000)
    smin = []
    for f in f_sweep:
        R = K - (2 * np.pi * f) ** 2 * M
        sv = np.linalg.svd(R, compute_uv=False)
        smin.append(sv.min() / sv.max())
    smin = np.array(smin)

    fig, ax = plt.subplots(figsize=(6.6, 4.0))
    ax.plot(f_sweep, smin, color=COLORS["primary"])
    for i, f0 in enumerate(fn):
        ax.axvline(f0, color=COLORS["secondary"], lw=0.8, ls=":")
        ax.text(f0, 1.02, rf"$f_{i+1}$", color=COLORS["secondary"],
                ha="center", va="bottom", fontsize=9,
                transform=ax.get_xaxis_transform())
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$\sigma_{\min}/\sigma_{\max}$ of $(K-\omega^2 M)$")
    ax.set_ylim(-0.03, 1.05)

    fig.tight_layout()
    save(fig, "fig_singularity_vs_freq")


# ===========================================================================
def main():
    apply_style()
    print("Generating figures:")
    fig_signal_transformation()
    fig_psd_damage_shift()
    fig_loss_comparison()
    fig_singularity_vs_freq()
    print("Done.")


if __name__ == "__main__":
    main()

Generating figures:
  saved fig_signal_transformation.{png,pdf}
  saved fig_psd_damage_shift.{png,pdf}
  saved fig_loss_comparison.{png,pdf}
    (Frobenius varies 2.7%, ratio varies 99.6%)
  saved fig_singularity_vs_freq.{png,pdf}
Done.
